In [0]:
%run ../01_setup/storage_config

In [0]:
# read data 
services_df = spark.table("ns_train_processed.services")
stops_df = spark.table("ns_train_processed.stops")

display(stops_df)

Select fields:
1. stop code
2. stop name
3. date
5. number of unique services
3. number of initial departures 
4. number of initial arrivals 
5. departure delay max daily
6. arrival delay max daily
7. long_delay_over_30
8. total calcellations


In [0]:
processed_stops_table = 'ns_train_processed.stops'

daily_summary_by_station_sql = f'''
WITH initial_stops_count AS (
SELECT
  service_date, 
  stop_station_name,
  service_rdt_id,
  CASE WHEN stop_arrival_time IS Null THEN 1 ELSE 0 END as is_starting_stop,
  CASE WHEN stop_departure_time IS Null THEN 1 ELSE 0 END as is_destination_stop,
  stop_arrival_delay,
  stop_departure_delay,
  CASE WHEN stop_arrival_delay >= 30 OR stop_departure_delay >= 30 THEN 1 ELSE 0 END as is_delayed_over_30,
  CASE WHEN stop_arrival_cancelled IS TRUE OR stop_departure_cancelled IS TRUE THEN 1 ELSE 0 END as is_cancelled
FROM {processed_stops_table}
)
SELECT 
  service_date, 
  stop_station_name,
  count(DISTINCT service_rdt_id) as number_of_unique_services,
  sum(is_starting_stop) as starting_station_count,
  sum(is_destination_stop) as destination_station_count,
  max(stop_departure_delay) as departure_delay_max,
  max(stop_arrival_delay) as arrival_delay_max,
  sum(is_delayed_over_30) as long_delay_over_30,
  sum(is_cancelled) as total_cancellations
FROM initial_stops_count
GROUP BY service_date, stop_station_name
ORDER BY service_date, number_of_unique_services DESC;
'''

daily_summary_by_station_df = spark.sql(daily_summary_by_station_sql)


In [0]:
folder_path = f'{presentation_folder_path}/{station_table}'
daily_summary_by_station_df.write.mode("overwrite").format("delta").save(folder_path)